# **0. Carga de datos**

In [1]:
import pandas as pd
import altair as alt
import numpy as np
alt.data_transformers.disable_max_rows()
df = pd.read_csv("data/cleaned_data.csv")

In [2]:
df

,string_id,aq30_id,name_0,name_1,area_km2,bws_score,bwd_score,iav_score,sev_score,drr_score,...,w_awr_fnb_tot_score,w_awr_min_tot_score,w_awr_ong_tot_score,w_awr_smc_tot_score,w_awr_tex_tot_score,continent,Riesgo_Hidrologico_total,Riesgo_Eventos_total,Riesgo_Calidad_Agua_total,Riesgo_por_Industria_total
0,111081-ERI.2_1-3365,89,Eritrea,Debub,6.445810,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.660862,4.421376,4.432915,4.644985,4.625870,Africa,3.890899,2.131406,4.412566,4.499972
1,111081-ERI.6_1-3365,90,Eritrea,Semenawi Keyih Bahri,210.189947,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.660862,4.421376,4.432915,4.644985,4.625870,Africa,3.890899,2.131406,4.412566,4.499972
2,111081-SDN.11_1-1775,92,Sudan,Red Sea,725.925119,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.410905,4.309340,4.314742,4.483448,4.459413,Africa,3.890899,2.131406,4.412566,4.355864
3,111081-SDN.11_1-1930,93,Sudan,Red Sea,345.262750,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.625538,4.374278,4.363119,4.606872,4.561502,Africa,3.890899,2.131406,4.412566,4.465850
4,111081-SDN.11_1-3365,94,Sudan,Red Sea,2989.133981,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.625538,4.374278,4.363119,4.606872,4.561502,Africa,3.890899,2.131406,4.412566,4.465850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46625,832808-CAN.3_1-352,61194,Canada,Manitoba,1657.645142,0.000000,0.001827,5.000000,2.576914,1.512279,...,0.754779,0.523294,0.309699,0.879260,0.637502,North America,1.818204,0.000000,0.678422,0.672614
46626,832808-CAN.8_1-352,61195,Canada,Nunavut,1468.762076,0.000000,0.001827,5.000000,2.576914,1.512279,...,0.754779,0.523294,0.309699,0.879260,0.637502,North America,1.818204,0.000000,0.678422,0.672614
46627,832809-CAN.12_1-352,61196,Canada,Saskatchewan,174.810225,0.000000,0.000025,3.387390,0.902210,1.473108,...,0.631738,0.412238,0.230250,0.705099,0.527545,North America,1.152547,0.000000,0.676915,0.532449
46628,832809-CAN.3_1-352,61197,Canada,Manitoba,7985.673792,0.000000,0.000025,3.387390,0.902210,1.473108,...,0.631738,0.412238,0.230250,0.705099,0.527545,North America,1.152547,0.000000,0.676915,0.532449


In [3]:
df.columns

Index(['string_id', 'aq30_id', 'name_0', 'name_1', 'area_km2', 'bws_score',
       'bwd_score', 'iav_score', 'sev_score', 'drr_score', 'rfr_score',
       'cfr_score', 'ucw_score', 'cep_score', 'udw_score', 'usa_score',
       'w_awr_agr_tot_score', 'w_awr_che_tot_score', 'w_awr_con_tot_score',
       'w_awr_elp_tot_score', 'w_awr_fnb_tot_score', 'w_awr_min_tot_score',
       'w_awr_ong_tot_score', 'w_awr_smc_tot_score', 'w_awr_tex_tot_score',
       'continent', 'Riesgo_Hidrologico_total', 'Riesgo_Eventos_total',
       'Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total'],
      dtype='str')

# **1. Ecuador en el contexto global del estrés hídrico**

## **Contexto de la pregunta analítica**

Para comprender la magnitud del reto hídrico en Ecuador, analizamos su posición dentro de la distribución mundial. El siguiente gráfico muestra el **Estrés Hídrico Base (Baseline Water Stress)** ponderado por área de todos los países, agrupados por continente. Esta vista panorámica permite contrastar la realidad ecuatoriana no solo a nivel global, sino frente al comportamiento de Sudamérica.

## **Visualización**

In [4]:
# Configuración global de estilo para Altair (Altair 5.5.0+)
alt.theme.enable('fivethirtyeight')

# 2. Función de promedio ponderado
def w_avg(df, values_cols, weight_col):
    d = df[values_cols]
    w = df[weight_col]
    return (d.multiply(w, axis=0).sum(axis=0)) / w.sum()

# 3. Agrupación provincial
cols_riesgo = [col for col in df.columns if col.endswith('_score') or col.endswith('_total')]
df_prov = df.groupby('name_1').apply(lambda x: w_avg(x, cols_riesgo, 'area_km2')).reset_index()

# 4. Dimensiones macro
cols_hidro = ['bws_score', 'bwd_score', 'iav_score', 'sev_score', 'drr_score']

cols_eventos = ['rfr_score', 'cfr_score']

cols_calidad = ['ucw_score', 'cep_score', 'udw_score', 'usa_score']

cols_industria = [col for col in df.columns if col.startswith('w_awr_')]

df_prov['Riesgo_Total'] = df[['Riesgo_Hidrologico_total', 'Riesgo_Eventos_total','Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total']].mean(axis=1)

In [34]:
import pandas as pd
import altair as alt
import numpy as np

# Configuración global
alt.theme.enable('fivethirtyeight')

# 1. Cargar el dataset global
df_global = pd.read_csv("data/cleaned_data_long_format.csv")

# 2. Filtrar únicamente el indicador de Estrés Hídrico Base
df_bws = df_global[df_global['score'] == 'bws_score'].copy()

# 3. Función de promedio ponderado global
def w_avg_global(df, val_col, weight_col):
    if df[weight_col].sum() == 0: 
        return 0
    return (df[val_col] * df[weight_col]).sum() / df[weight_col].sum()

# 4. Agrupar la data para calcular el score final de cada PAÍS del mundo
df_paises = df_bws.groupby(['name_0', 'continent']).apply(
    lambda x: w_avg_global(x, 'value', 'area_km2')
).reset_index(name='Estres_Hidrico')

df_paises['continent'] = df_paises['continent'].replace({'South America': 'Sudamérica', 'North America': 'Norteamérica'})

# Crear 'Jitter' (ruido vertical) para evitar solapamiento
np.random.seed(42)
df_paises['jitter'] = np.random.uniform(-0.0, 0.0, len(df_paises))

# 5. Construcción de la Visualización
base = alt.Chart(df_paises).encode(
    x=alt.X('Estres_Hidrico:Q', title='Estrés Hídrico Base (0-5)', scale=alt.Scale(domain=[-0.2, 5.2])),
    y=alt.Y('continent:N', title='Continente', sort='ascending', axis=alt.Axis(grid=True, labelFontSize=12))
)

# Capa 1: Puntos
puntos = base.mark_circle(stroke='white', strokeWidth=0.5).encode(
    yOffset='jitter:Q', 
    
    # ---------------------------------------------------------
    # LA MAGIA ARQUITECTÓNICA: Condición con Escala
    # Si es Ecuador -> Rojo Intenso. Si NO -> Color por Continente (paleta set2)
    # ---------------------------------------------------------
    color=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value('#cc0000'),  # Valor fijo para Ecuador (Rojo Intenso)
        alt.Color('continent:N', scale=alt.Scale(scheme='set2'), legend=None) # Escala de colores para el resto
    ),
    
    size=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value(300),          # Hacemos a Ecuador aún más grande para que domine
        alt.value(70)
    ),
    opacity=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value(1.0),          # Ecuador sólido
        alt.value(0.7)           # Resto de países ligeramente transparentes
    ),
    tooltip=[
        alt.Tooltip('name_0:N', title='País'),
        alt.Tooltip('continent:N', title='Continente'),
        alt.Tooltip('Estres_Hidrico:Q', title='Estrés Hídrico', format='.2f')
    ]
)

# Capa 2: Etiqueta de Ecuador
texto_ecuador = base.mark_text(
    align='left', 
    baseline='middle', 
    dx=14,           
    fontSize=13, 
    fontWeight='bold', 
    color='#cc0000' # Mismo color que el punto
).transform_filter(
    alt.datum.name_0 == 'Ecuador'
).encode(
    yOffset='jitter:Q', 
    text='name_0:N'
)

# Capa 3: Línea y etiqueta de promedio mundial
promedio_mundial = df_paises['Estres_Hidrico'].mean()

linea_promedio = alt.Chart(pd.DataFrame({'x': [promedio_mundial]})).mark_rule(
    strokeDash=[5, 5], color='#4A4A4A', strokeWidth=1.5
).encode(x='x:Q')

texto_promedio = alt.Chart(pd.DataFrame({'x': [promedio_mundial]})).mark_text(
    align='left', baseline='bottom', dx=5, dy=-5, 
    text='Promedio Mundial', color='#4A4A4A', fontSize=11, fontWeight='bold'
).encode(
    x='x:Q', 
    y=alt.value(0) 
)

# Ensamblaje final
grafico_contexto = (puntos + texto_ecuador + linea_promedio + texto_promedio).properties(
    title=alt.TitleParams(
        text='Posicionamiento Global del Estrés Hídrico',
        subtitle='Cada punto representa un país.',
        color='#333333',
        dy=-10
    ),
    width=650, 
    height=350
).interactive()

grafico_contexto

alt.LayerChart(...)

In [31]:
import pandas as pd
import altair as alt
import numpy as np

# Configuración global
alt.theme.enable('fivethirtyeight')

# 1. Asumiendo que df_bws ya está cargado y filtrado
# df_bws = df_global[df_global['score'] == 'bws_score'].copy()

def w_avg_global(df, val_col, weight_col):
    if df[weight_col].sum() == 0: return 0
    return (df[val_col] * df[weight_col]).sum() / df[weight_col].sum()

# 2. Agrupación y cálculo del Promedio Mundial
df_paises = df_bws.groupby(['name_0', 'continent']).apply(
    lambda x: w_avg_global(x, 'value', 'area_km2')
).reset_index(name='Estres_Hidrico')

df_paises['continent'] = df_paises['continent'].replace({'South America': 'Sudamérica', 'North America': 'Norteamérica'})

promedio_mundial = (df_bws['value'] * df_bws['area_km2']).sum() / df_bws['area_km2'].sum()
df_paises['prom_mundial'] = promedio_mundial

# 3. Jitter para los puntos
np.random.seed(42)
df_paises['jitter'] = np.random.uniform(0, 0.4, len(df_paises))

# --- CAPAS DEL RAINCLOUD ---

# Capa 1: NUBE (Densidad)
cloud = alt.Chart().transform_density(
    density='Estres_Hidrico', groupby=['continent'], steps=200, extent=[-0.2, 5.2], as_=['Estres_Hidrico', 'density']
).mark_area(orient='vertical', opacity=0.3).encode(
    x=alt.X('Estres_Hidrico:Q', title='Estrés Hídrico Base (0 = Bajo, 5 = Extremo)', scale=alt.Scale(domain=[-0.2, 5.2])),
    y=alt.Y('density:Q', title=None, axis=None, scale=alt.Scale(range=[30, 0])), 
    color=alt.Color('continent:N', legend=None, scale=alt.Scale(scheme='set2'))
)

# Capa 2: PARAGUAS (Boxplot). Se ubica en una posición vertical fija (y=35) entre la nube y la lluvia
umbrella = alt.Chart().mark_boxplot(
    size=12, color='#333333', opacity=0.7, outliers=False # Ocultamos outliers porque ya se verán en la lluvia
).encode(
    x='Estres_Hidrico:Q',
    y=alt.value(35) # Posición vertical fija
)

# Capa 3: LLUVIA (Puntos)
rain = alt.Chart().mark_circle(stroke='white', strokeWidth=0.5).encode(
    x='Estres_Hidrico:Q',
    y=alt.Y('jitter:Q', title=None, axis=None, scale=alt.Scale(range=[45, 65])),
    color=alt.condition(
        alt.datum.name_0 == 'Ecuador', alt.value('#cc0000'), alt.Color('continent:N', scale=alt.Scale(scheme='set2'), legend=None)
    ),
    size=alt.condition(alt.datum.name_0 == 'Ecuador', alt.value(200), alt.value(50)),
    opacity=alt.condition(alt.datum.name_0 == 'Ecuador', alt.value(1.0), alt.value(0.6)),
    tooltip=[alt.Tooltip('name_0:N', title='País'), alt.Tooltip('Estres_Hidrico:Q', title='Score', format='.2f')]
)

# Capa 4: Etiqueta Ecuador
texto_ecuador = alt.Chart().mark_text(
    align='left', baseline='middle', dx=10, dy=0, fontSize=13, fontWeight='bold', color='#cc0000'
).transform_filter(alt.datum.name_0 == 'Ecuador').encode(
    x='Estres_Hidrico:Q', y=alt.Y('jitter:Q', scale=alt.Scale(range=[45, 65])), text='name_0:N'
)

# Capa 5 y 6: Promedio Mundial y su Etiqueta
linea_promedio = alt.Chart().mark_rule(strokeDash=[5, 5], color='#4A4A4A', strokeWidth=1.5).encode(x='prom_mundial:Q')

texto_promedio = alt.Chart().mark_text(
    align='left', baseline='bottom', dx=5, dy=-15, color='#4A4A4A', fontSize=11, fontWeight='bold'
).transform_aggregate(
    prom_mundial='min(prom_mundial)', groupby=['continent']
).encode(
    x='prom_mundial:Q', y=alt.value(0), text=alt.Text('prom_mundial:Q', format='.2f')
)

# Ensamblaje con Facet
grafico_final = alt.layer(
    cloud, umbrella, rain, texto_ecuador, linea_promedio, texto_promedio, data=df_paises
).properties(
    width=650, height=80 
).facet(
    row=alt.Row('continent:N', title=None, sort='ascending', header=alt.Header(labelAngle=0, labelAlign='left', labelFontSize=13, labelFontWeight='bold'))
).properties(
    title=alt.TitleParams(
        text='Ecuador en el Contexto Global del Estrés Hídrico',
        subtitle=['Cada punto es un país. La "nube" muestra dónde se concentra la mayoría.', 'La caja negra indica el rango donde está el 50% de los países de cada continente.'],
        color='#333333', dy=-10
    )
).resolve_scale(y='independent')
#.interactive(bind_y=False)

grafico_final.show()

alt.FacetChart(...)

## **Interpretación**

### 📊 1. Interpretación Analítica: Ecuador en el contexto global del estrés hídrico

Esta visualización parte de un Un **Strip Plot** (o gráfico de franjas), un tipo de gráfico estadístico que representa la distribución de datos numéricos individuales como puntos a lo largo de un eje. Posteriomente se desarrolla como un **Raincloud Plot** (Gráfico de Nube de Lluvia), una técnica analítica avanzada que ofrece una perspectiva superior al clásico *Boxplot*. Permite observar simultáneamente la densidad de probabilidad (KDE), los estadísticos de resumen (caja) y la distribución real de los puntos geográficos (scatter).

**Insights:**

* **La posición de Ecuador:** La línea punteada roja sitúa a Ecuador en el extremo izquierdo de la escala (aproximadamente < 0.5 en un rango de 0 a 5). A nivel macro, esto indica que el país goza de un **Estrés Hídrico Base (BWS) extremadamente bajo**. La disponibilidad natural de agua dulce supera de manera holgada a la demanda agregada en la mayor parte de su territorio.
* **El perfil de Sudamérica:** La distribución de la región presenta una fuerte asimetría hacia la derecha (sesgo positivo). La "nube" principal de datos se concentra cerca del 0, fuertemente influenciada por la cuenca amazónica y los Andes. Sin embargo, la cola extendida revela zonas específicas con estrés alto crónico (como el norte de Chile). Ecuador se encuentra cómodamente alineado con la moda estadística de su continente.
* **Contraste Global:** Al observar Asia y África, se evidencian distribuciones mucho más uniformes o incluso bimodales, revelando enormes extensiones territoriales que sufren de estrés hídrico crítico (valores entre 4 y 5). Frente a este panorama mundial, Ecuador representa un "oasis" comparativo en términos de disponibilidad base de agua.

* _**Nota:** El hecho de que el "Estrés Base" sea bajo no significa que Ecuador no tenga problemas de agua. Muchas veces, los verdaderos riesgos están en la variabilidad interanual (iav_score) o en el riesgo de eventos extremos como inundaciones y sequías, fenómenos muy relevantes en la geografía andina y costera._

# **2. Riesgo hidrológico por provincia**

## **Contexto de la pregunta analítica**

El análisis previo demostró que, a nivel global, Ecuador posee un estrés hídrico base muy bajo. Sin embargo, el "Riesgo Hidrológico" es un constructo multidimensional. Plantearnos qué provincias tienen mayor riesgo y qué indicadores lo impulsan nos permite pasar de una estadística descriptiva general a una **analítica prescriptiva**. Esto es fundamental para la asignación de recursos públicos, diseño de políticas de infraestructura y mitigación de desastres focalizados (por ejemplo, saber si una provincia necesita plantas de tratamiento de agua versus muros de contención para inundaciones).

**Preguntas analíticas:**
- ¿Qué provincias del Ecuador presentan mayores scores de riesgo hidrológico?
- ¿Qué indicadores hidrológicos destacan más en cada provincia?


## **Visualización**

In [ ]:
# 5. Visualización Interactiva con Filtro Base Cero
seleccion_indicador = alt.selection_point(fields=['Indicador'], bind='legend')

# Construimos la base de las barras
bars_hidro = alt.Chart(df_hidro_melt).mark_bar(
    cornerRadiusEnd=3, 
    stroke='white',    
    strokeWidth=0.5
).encode(
    x=alt.X('sum(Score):Q', title='Score Acumulado (Anclado a Cero)'),
    
    # IMPORTANTE: El sort se recalculará de forma dinámica al filtrar
    y=alt.Y('name_1:N', sort=alt.EncodingSortField(field='Score', op='sum', order='descending'), title=None),
    
    # IMPORTANTE: Fijamos el 'domain' explícitamente usando la lista para que la leyenda no desaparezca
    color=alt.Color('Indicador:N', 
                    scale=alt.Scale(scheme='reds', domain=lista_indicadores),
                    legend=alt.Legend(title="Indicadores (Clic)", symbolType='circle', orient='top')),
    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Indicador:N', title='Sub-indicador'),
        alt.Tooltip('Score:Q', title='Score', format='.2f')
    ]
).properties(
    width=600, 
    height=alt.Step(25) # Mantiene el grosor de la barra uniforme
).add_params(
    seleccion_indicador
).transform_filter(
    seleccion_indicador # <--- LA MAGIA: Esto hace que los datos no seleccionados se eliminen y las barras caigan a cero
)

# ENSAMBLAJE FINAL: Aplicar Facet por Región
grafico_facetado = bars_hidro.facet(
    row=alt.Row('Region:N', 
                title=None, 
                header=alt.Header(labelFontSize=14, labelFontWeight='bold', labelAngle=0, labelAlign='' \
                ''))
).resolve_scale(
    y='independent'
).properties(
    title=alt.TitleParams(
        text='Radiografía del Riesgo Hidrológico por Provincia y Región',
        subtitle=['Clasificado por Región Natural.', '💡 Interactividad: Haz clic en la leyenda. Las barras se apilarán desde el eje cero y el ranking se recalculará automáticamente.'],
        color='#333333',
        dy=-10
    )
)

grafico_facetado.show()

alt.FacetChart(...)

## **Interpretación**

### 📊 2. Interpretación Analíticas
El desglose del riesgo hidrológico revela que la vulnerabilidad hídrica en Ecuador es un fenómeno multidimensional y altamente focalizado geográficamente. Al pasar de una vista global a una provincial, podemos extraer los siguientes hallazgos prescriptivos:

**1. ¿Qué provincias del Ecuador presentan mayores scores de riesgo hidrológico?**
A nivel nacional, el riesgo no se distribuye de manera uniforme. Las provincias con mayor criticidad se concentran en la región Litoral y el sur de la región Interandina:
* **Loja (Sierra):** Lidera el ranking nacional con el score acumulado más alto de todo el país (aprox. 11.04), convirtiéndose en el principal foco de atención.
* **El bloque de la Costa:** Provincias como **El Oro, Santa Elena, Manabí y Guayas** ocupan los siguientes lugares de mayor riesgo, dominando el perfil de vulnerabilidad de la región litoral.
* **Contraste Amazónico:** En el extremo opuesto, la región de la Amazonía (provincias como Napo, Orellana o Sucumbíos) presenta los niveles mínimos de riesgo hidrológico, actuando como la gran reserva estable del país.

**2. ¿Qué indicadores hidrológicos destacan más en cada provincia?**
Al descomponer las barras, descubrimos que el principal desafío de las provincias más críticas **no es la escasez absoluta o constante de agua** (*Estrés Base*), sino la alta **inestabilidad climática**:
* **El caso de Loja:** Su posición número uno está impulsada casi en su totalidad por la **Variabilidad Interanual** (cambios drásticos de disponibilidad de agua de un año a otro) y el **Riesgo de Sequía**.

# **3. Eventos hidrológicos**

## **Contexto de la pregunta analítica**

### 📊 3. Interpretación Analítica: Análisis Bivariado de Riesgo de Inundaciones

**Contexto Analítico:**
Mientras que la visualización anterior nos mostraba el riesgo acumulado, este gráfico de dispersión (*scatter plot*) nos permite realizar un **análisis bivariado**. Entender la interacción entre el riesgo fluvial (ríos) y el riesgo costero (océano) es vital para la planificación de políticas públicas y presupuestos. Desde una perspectiva de ingeniería, las obras de mitigación son diametralmente distintas: no es lo mismo invertir en dragado de ríos y muros de contención (fluvial) que en conservación de manglares y rompeolas (costero).

**Preguntas analíticas:**
- ¿Qué provincias presentan mayores scores asociados a inundaciones fluviales y costeras?
- ¿Existen diferencias claras entre provincias en la exposición a estos eventos?

## **Visualización**

In [46]:
# =========================================================================
# --- CELDA ACTUALIZADA: Dumbbell Plot (Fluvial vs Costera) ---
# =========================================================================

# 1. Melt: Añadimos 'Riesgo_Eventos_total' a id_vars para usarlo como criterio de orden
df_eventos_melt = df_prov.melt(
    id_vars=['name_1', 'Riesgo_Eventos_total'], 
    value_vars=['rfr_score', 'cfr_score'], 
    var_name='Tipo_Inundacion', 
    value_name='Score'
)

# 2. Renombrar para legibilidad en la visualización
df_eventos_melt['Tipo_Inundacion'] = df_eventos_melt['Tipo_Inundacion'].replace({
    'rfr_score': 'Fluvial (Ríos)', 
    'cfr_score': 'Costera'
})

# 3. Construcción del Dumbbell Plot
# CAPA 1: La línea que conecta los puntos (la "barra" de la pesa)
rule = alt.Chart(df_eventos_melt).mark_rule(color='#c0c0c0', strokeWidth=2.5).encode(
    x=alt.X('min(Score):Q', title='Score de Riesgo (0-5)', scale=alt.Scale(domain=[0, 5])),
    x2=alt.X2('max(Score):Q'),
    
    # Ordenamos el Eje Y basándonos en el Riesgo Total de Eventos (descendente)
    y=alt.Y('name_1:N', 
            sort=alt.EncodingSortField(field='Riesgo_Eventos_total', op='max', order='descending'), 
            title='Provincia', 
            axis=alt.Axis(grid=True)) # Añadimos grid horizontal para guiar el ojo
)

# CAPA 2: Los puntos (las "pesas")
points = alt.Chart(df_eventos_melt).mark_circle(size=180, opacity=1).encode(
    x=alt.X('Score:Q'),
    y=alt.Y('name_1:N', sort=alt.EncodingSortField(field='name_1', op='max', order='descending')),
    
    color=alt.Color('Tipo_Inundacion:N', 
                    scale=alt.Scale(domain=['Fluvial (Ríos)', 'Costera'], range=['#1f77b4', '#ff7f0e']),
                    legend=alt.Legend(title="Tipo de Inundación", orient='top-right', fillColor='white', padding=10)),
    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Tipo_Inundacion:N', title='Tipo de Inundación'),
        alt.Tooltip('Score:Q', title='Score Específico', format='.2f'),
        # Al incluir el total en el tooltip, damos más contexto al usuario
        alt.Tooltip('Riesgo_Eventos_total:Q', title='Riesgo Total de Eventos', format='.2f') 
    ]
)

# Ensamblaje final
dumbbell = (rule + points).properties(
    title=alt.TitleParams(
        text='Brecha de Riesgo: Inundaciones Fluviales vs. Costeras',
        subtitle='Provincias ordenadas por su exposición total a eventos extremos.',
        color='#333333'
    ),
    width=650, 
    height=550
).interactive()

# Mostrar gráfico
dumbbell

alt.LayerChart(...)

## **Interpretación**

* **¿Qué provincias presentan mayores scores asociados a inundaciones fluviales y costeras?**
  * **Riesgo Combinado:** **Guayas y Santa Elena** son las provincias con la situación más crítica, liderando de forma absoluta en ambos ejes (con puntajes cercanos al máximo de 5.0). Le siguen de cerca **El Oro, Esmeraldas y Manabí**, las cuales presentan una alta vulnerabilidad simultánea a desbordamientos de ríos y mareajes.
  * **Riesgo Fluvial Puro:** **Los Ríos** destaca fuertemente; aunque su riesgo costero es nulo (0), su exposición a inundaciones fluviales es la más alta del país junto con Guayas (aprox. 4.8).

* **¿Existen diferencias claras entre provincias en la exposición a estos eventos?**
  * Sí, las diferencias son radicales y forman **clústeres geográficos naturales**. 
  * Existe una clara "línea base" en el eje Y (Riesgo Costero = 0) donde se agrupa la inmensa mayoría de las provincias de la Sierra y la Amazonía (Pichincha, Azuay, Tungurahua, Napo, etc.). Para estas provincias, la única dimensión de riesgo de inundación es la fluvial, la cual varía moderadamente entre 0.5 y 3.0.
  * Por otro lado, las provincias de la región Costa se dispersan en ambos ejes, demostrando que su perfil de riesgo hidrológico es mucho más complejo y requiere estrategias de mitigación multidimensionales.

# **4. Calidad del agua**

## **Contexto de la pregunta analítica**

Si bien el análisis anterior nos permitió diagnosticar los riesgos asociados a la disponibilidad física del recurso (sequías y variabilidad climática), la seguridad hídrica de una región no está completa sin evaluar su dimensión sanitaria. Tener agua suficiente es apenas el primer paso; garantizar que sea apta para el consumo humano y que el entorno cuente con servicios de saneamiento adecuados es vital. 

Para entender esta brecha, evaluamos cuatro indicadores clave: exposición a aguas residuales no tratadas, eutrofización costera, acceso a agua potable segura y saneamiento. A través del siguiente mapa de calor, buscamos responder:
* ¿Qué provincias presentan mayores vulnerabilidades en calidad del agua y acceso a servicios básicos?
* ¿Qué dimensiones específicas arrastran los perfiles más críticos del país?

## **Visualización**

In [41]:
import pandas as pd
import altair as alt

# Configuración global
alt.data_transformers.disable_max_rows()
alt.theme.enable('fivethirtyeight')

# 1. Cargar datos y filtrar Ecuador
df = pd.read_csv("data/cleaned_data.csv")
df_ecuador = df[df['name_0'] == 'Ecuador'].copy()

# 2. Función de promedio ponderado
def w_avg(df_part, values_cols, weight_col):
    d = df_part[values_cols]
    w = df_part[weight_col]
    return (d.multiply(w, axis=0).sum(axis=0)) / w.sum()

# 3. Agrupación provincial solo para calidad de agua
cols_calidad = ['ucw_score', 'cep_score', 'udw_score', 'usa_score']
df_prov = df_ecuador.groupby('name_1').apply(lambda x: w_avg(x, cols_calidad, 'area_km2')).reset_index()

# Calculamos un "Score Total de Calidad" para ordenar el gráfico de los más críticos a los menos
df_prov['Total_Calidad'] = df_prov[cols_calidad].sum(axis=1)

# 4. Mapeo de Regiones Naturales
mapa_regiones = {
    'Esmeraldas': '1. Costa', 'Manabi': '1. Costa', 'Los Rios': '1. Costa', 'Guayas': '1. Costa', 
    'Santa Elena': '1. Costa', 'El Oro': '1. Costa', 'Santo Domingo de los Tsachilas': '1. Costa',
    'Carchi': '2. Sierra', 'Imbabura': '2. Sierra', 'Pichincha': '2. Sierra', 'Cotopaxi': '2. Sierra', 
    'Tungurahua': '2. Sierra', 'Bolivar': '2. Sierra', 'Chimborazo': '2. Sierra', 'Cañar': '2. Sierra', 
    'Azuay': '2. Sierra', 'Loja': '2. Sierra',
    'Sucumbios': '3. Amazonía', 'Napo': '3. Amazonía', 'Orellana': '3. Amazonía', 'Pastaza': '3. Amazonía', 
    'Morona Santiago': '3. Amazonía', 'Zamora Chinchipe': '3. Amazonía'
}
df_prov['Region'] = df_prov['name_1'].map(mapa_regiones)

# 5. Transformación Melt para el Heatmap
df_calidad_melt = df_prov.melt(
    id_vars=['name_1', 'Region', 'Total_Calidad'], 
    value_vars=cols_calidad, 
    var_name='Indicador', 
    value_name='Score'
)

# Diccionario de traducción y numeración para forzar orden
dict_calidad = {
    'ucw_score': '1. Aguas Residuales no tratadas', 
    'cep_score': '2. Eutrofización Costera', 
    'udw_score': '3. Agua Potable Insegura', 
    'usa_score': '4. Saneamiento Inseguro'
}
df_calidad_melt['Indicador'] = df_calidad_melt['Indicador'].map(dict_calidad)

# 6. Construcción del Heatmap
heatmap = alt.Chart(df_calidad_melt).mark_rect(
    stroke='white', 
    strokeWidth=1.5 # Líneas blancas entre los cuadros para dar efecto de "baldosa" limpia
).encode(
    x=alt.X('Indicador:N', 
            title=None, 
            axis=alt.Axis(labelAngle=-45, labelAlign='right', labelFontSize=12, labelLimit=300)),
    
    # Ordenamos las provincias de mayor riesgo a menor riesgo DENTRO de cada región
    y=alt.Y('name_1:N', 
            title=None, 
            sort=alt.EncodingSortField(field='Total_Calidad', op='max', order='descending'),
            axis=alt.Axis(labelFontSize=12)),
    
    # Usamos una paleta "orangered" (secuencial de advertencia)
    color=alt.Color('Score:Q', 
                    scale=alt.Scale(scheme='orangered', domain=[0, 5]), 
                    legend=alt.Legend(title='Nivel de Riesgo (0-5)', gradientLength=200)),
    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Indicador:N', title='Indicador'),
        alt.Tooltip('Score:Q', title='Score', format='.2f')
    ]
).properties(
    width=350,
    height=alt.Step(25) # Altura dinámica para que los cuadrados sean proporcionados
)

# Ensamblaje con Facet por región
grafico_calidad = heatmap.facet(
    row=alt.Row('Region:N', title=None, header=alt.Header(labelFontSize=14, labelFontWeight='bold', labelAngle=0, labelAlign='left'))
).resolve_scale(
    y='independent'
).properties(
    title=alt.TitleParams(
        text='Vulnerabilidad en Calidad del Agua y Servicios Básicos',
        subtitle=['Colores más oscuros indican un mayor riesgo (peores condiciones).', 'Las provincias están ordenadas por su nivel de criticidad general en cada región.'],
        color='#333333',
        dy=-10
    )
)

grafico_calidad.show()

alt.FacetChart(...)

## **Interpretación**

### Interpretación: Vulnerabilidad en Calidad del Agua y Servicios

El mapa de calor nos permite diagnosticar de forma simultánea qué regiones carecen de infraestructura sanitaria adecuada y qué factores impulsan este riesgo. Respondiendo a las interrogantes planteadas:

**1. ¿Qué provincias presentan mayores scores en indicadores relacionados con calidad del agua y acceso a servicios básicos?**
Al observar la intensidad de los colores y el ranking de criticidad (orden descendente por región):
* **Loja (Sierra):** Emerge nuevamente como el territorio más vulnerable del país. Sus celdas oscuras en *Agua Potable Insegura* (4.11) y *Saneamiento Inseguro* (4.24) sugieren deficiencias críticas en la provisión de servicios básicos.
* **Provincias Amazónicas:** Contrario a la abundancia hídrica natural de la región, provincias como **Zamora Chinchipe, Pastaza y Sucumbios** muestran puntajes alarmantemente altos (cercanos a 4.0) en la carencia de *Agua Potable Segura*. Esto evidencia una paradoja territorial: la Amazonía tiene agua en abundancia, pero carece de infraestructura de potabilización.

**2. ¿Qué provincias muestran perfiles más críticos en dimensiones específicas?**
El análisis por columnas revela los "talones de Aquiles" sanitarios del Ecuador:
* **Eutrofización Costera (Contaminación por nutrientes):** Como es de esperarse geográficamente, este riesgo afecta exclusivamente a la región Litoral, pero **Loja** y **El Oro** destacan como las zonas de mayor alerta. Las descargas agrícolas y de aguas residuales en las cuencas del sur del país están deteriorando severamente la calidad del agua hacia las desembocaduras.
* **Saneamiento Inseguro y Aguas Residuales:** El indicador de *Aguas Residuales no Tratadas* presenta un riesgo elevado y constante en todo el país (3.72), reflejando un déficit estructural a nivel nacional en plantas de tratamiento (PTAR). 

**Acción Prescriptiva:** La política pública no debe enfocarse en "buscar agua" en el sur y la Amazonía, sino en financiar urgentemente plantas de potabilización y alcantarillado sanitario para mitigar riesgos de salud pública.

# **5. Riesgo por industria**

## **Contexto de la pregunta analítica**

Para identificar qué provincias concentran altos niveles de riesgo en múltiples dimensiones y sectores, empleamos **Mapas de Calor (Heatmaps)**. Esto nos permite una lectura matricial rápida de los territorios más críticos.

## **Visualización**

In [48]:
# =========================================================================
# --- CELDA ACTUALIZADA: Heatmap de Riesgo por Sector Industrial ---
# =========================================================================

# 1. Melt: Añadimos 'Riesgo_por_Industria_total' a id_vars para usarlo como criterio de orden
df_ind_melt = df_prov.melt(
    id_vars=['name_1', 'Riesgo_por_Industria_total'], 
    value_vars=cols_industria, 
    var_name='Sector', 
    value_name='Score'
)

# 2. Diccionario y mapeo para legibilidad
dict_sectores = {
    'w_awr_agr_tot_score': 'Agricultura', 
    'w_awr_che_tot_score': 'Química', 
    'w_awr_con_tot_score': 'Construcción',
    'w_awr_elp_tot_score': 'Energía', 
    'w_awr_fnb_tot_score': 'Alimentos', 
    'w_awr_min_tot_score': 'Minería',
    'w_awr_ong_tot_score': 'Petróleo/Gas', 
    'w_awr_smc_tot_score': 'Semiconductores', 
    'w_awr_tex_tot_score': 'Textil'
}
df_ind_melt['Sector'] = df_ind_melt['Sector'].map(dict_sectores)

# 3. Construcción del Heatmap
heat_ind = alt.Chart(df_ind_melt).mark_rect(
    stroke='white',      # Añade una fina línea blanca entre celdas (Best Practice)
    strokeWidth=0.5
).encode(
    x=alt.X('Sector:N', 
            title='Sector Económico', 
            axis=alt.Axis(labelAngle=-45, labelFontSize=11)),
            
    # Ordenamos el Eje Y basándonos en el Riesgo Industrial Total (descendente)
    y=alt.Y('name_1:N', 
            sort=alt.EncodingSortField(field='Riesgo_por_Industria_total', op='max', order='descending'), 
            title='Provincia',
            axis=alt.Axis(labelFontSize=11)),
    
    # Mantenemos tu excelente decisión de anclar el domain=[0, 5]
    color=alt.Color('Score:Q', 
                    scale=alt.Scale(scheme='inferno', reverse=True, domain=[0, 5]), 
                    title='Riesgo (0-5)',
                    legend=alt.Legend(orient='right', padding=10)),
                    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Sector:N', title='Sector Económico'),
        alt.Tooltip('Score:Q', title='Score Sectorial', format='.2f'),
        # Añadimos el promedio global industrial al tooltip para dar más contexto
        alt.Tooltip('Riesgo_por_Industria_total:Q', title='Riesgo Industrial Promedio', format='.2f')
    ]
).properties(
    title=alt.TitleParams(
        text='Riesgo Hídrico por Sector Industrial',
        subtitle='Provincias ordenadas por su exposición promedio total del sector industria.',
        color='#333333'
    ),
    width=550,  # Un poco más ancho para que los nombres inclinados respiren bien
    height=550  # Un poco más alto para que las celdas sean más cuadradas
).interactive()

# Mostrar el gráfico
heat_ind

alt.Chart(...)

## **Interpretación**

# **6. Integración del análisis**

## **Contexto de la pregunta analítica**

## **Visualización**

In [52]:
df.columns

Index(['string_id', 'aq30_id', 'name_0', 'name_1', 'area_km2', 'bws_score',
       'bwd_score', 'iav_score', 'sev_score', 'drr_score', 'rfr_score',
       'cfr_score', 'ucw_score', 'cep_score', 'udw_score', 'usa_score',
       'w_awr_agr_tot_score', 'w_awr_che_tot_score', 'w_awr_con_tot_score',
       'w_awr_elp_tot_score', 'w_awr_fnb_tot_score', 'w_awr_min_tot_score',
       'w_awr_ong_tot_score', 'w_awr_smc_tot_score', 'w_awr_tex_tot_score',
       'continent', 'Riesgo_Hidrologico_total', 'Riesgo_Eventos_total',
       'Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total'],
      dtype='object')

In [53]:
bubble_integrated = alt.Chart(df_prov).mark_circle(stroke='white', strokeWidth=1, opacity=0.8).encode(
    x=alt.X('Riesgo_Hidrologico_total:Q', title='Riesgo Hidrológico', scale=alt.Scale(zero=False)),
    y=alt.Y('Riesgo_Calidad_Agua_total:Q', title='Riesgo en Calidad de Agua', scale=alt.Scale(zero=False)),
    size=alt.Size('Riesgo_Eventos_total:Q', title='Eventos Extremos', scale=alt.Scale(range=[50, 800])),
    color=alt.Color('Riesgo_por_Industria_total:Q', title='Riesgo Industrial', scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Riesgo_Total:Q', title='Riesgo Global Promedio', format='.2f'),
        alt.Tooltip('Riesgo_Hidrologico_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_Calidad_Agua_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_Eventos_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_por_Industria_total:Q', format='.2f')
    ]
).properties(
    title='Visión Integrada: Las 4 Dimensiones del Riesgo Hídrico',
    width=650, height=500
)

# Añadimos cuadrantes basados en la mediana para perfilar
vline = alt.Chart(pd.DataFrame({'x': [df_prov['Riesgo_Hidrologico_total'].median()]})).mark_rule(strokeDash=[3,3]).encode(x='x:Q')
hline = alt.Chart(pd.DataFrame({'y': [df_prov['Riesgo_Calidad_Agua_total'].median()]})).mark_rule(strokeDash=[3,3]).encode(y='y:Q')

(bubble_integrated + vline + hline).interactive()

alt.LayerChart(...)

## **Interpretación**